# Formation Energy and DOS of the V_Sn-O_i Defect Pair in SnO

> **A. Togo, F. Oba, and I. Tanaka**
> "First-principles calculations of native defects in tin monoxide"
> Phys. Rev. B 74, 195128 (2006)
> [DOI: 10.1103/PhysRevB.74.195128](https://doi.org/10.1103/PhysRevB.74.195128)

Computes the neutral formation energy of the V_Sn-O_i pair created in the
[structure notebook](defect_point_interstitial_tin_oxide.ipynb) and compares it with Togo et al.'s
Table II, model (a), 2.3 eV. Table II does not name the chemical-potential limit it uses, so E_f is
computed at both of them. Total Energy jobs on the pristine supercell, α-Sn and SnO₂ supply the
chemical potentials; a Density of States job on the pair cell supplies both its own `total_energy`
and the secondary check on in-gap states.

<h2 style="color:green">Usage</h2>

1. Create the materials in the [structure notebook](defect_point_interstitial_tin_oxide.ipynb), which saves `SnO 2x2x2 supercell` and `SnO 2x2x2 V_Sn-O_i pair (Togo Fig 4a)` to the `uploads` folder.
1. Set the material names and parameters in cells 1.2-1.4 (or use the defaults); `RELAX = True` uses the relaxed pair, relaxing once and saving it as `<name> relaxed` for reuse by this and other notebooks.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, model and compute parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the pristine and pair materials by name, load α-Sn and SnO₂ from Standata, print provenance and the V_Sn-O_i distance, then save all four to the platform.
1. Configure the shared DFT model and k-grid: one model and a per-material k-grid for every workflow below.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Create the missing Total Energy prerequisite jobs (pristine, α-Sn, SnO₂) and wait for them.
1. Relax the pair cell if `RELAX`, reusing a saved relaxed structure when one already exists.
1. Configure, create, submit and monitor the Density of States job on the pair cell.
1. Retrieve the results: the total energies, the chemical potentials at the Sn-rich and O-rich limits and the formation energy at each, then the pair's density of states and band gap.
1. Compare the formation energy with Togo et al. (2006).


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples|api_examples")


### 1.2. Material names

In [ ]:
# Names saved by defect_point_interstitial_tin_oxide.ipynb.
PRISTINE_NAME = "SnO 2x2x2 supercell"
DEFECTIVE_NAME = "SnO 2x2x2 V_Sn-O_i pair (Togo Fig 4a)"
# Standata references fixing the chemical potentials (Togo §II): α-Sn the Sn-rich limit, SnO₂ the O-rich limit.
SN_REFERENCE_NAME = "Sn, Tin, FCC (Fd-3m) 3D (Bulk), mp-117"
SNO2_REFERENCE_NAME = "SnO2, Tin Dioxide, TET (P4_2/mnm) 3D (Bulk), mp-856"


### 1.3. Parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"

TOTAL_ENERGY_SEARCH_TERM = "total_energy.json"
RELAX_WORKFLOW_SEARCH_TERM = "fixed_cell_relaxation.json"
DOS_SEARCH_TERM = "dos.json"
APPLICATION_NAME = "espresso"

# False: use the structure as given, fast. True: use the relaxed pair, running the
# relaxation once if it does not exist yet.
RELAX = False

CLUSTER_NAME = None
QUEUE_NAME = QueueName.OF
PPN = 40
TIME_LIMIT = "12:00:00"  # covers the optional relaxation

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds


### 1.4. DFT model parameters

In [ ]:
FUNCTIONAL = "pbe"
PSEUDOPOTENTIAL_TYPE = "us"  # GBRV ultrasoft, the platform's default PBE family for Sn and O
ECUTWFC = 40   # Ry, GBRV's recommended wavefunction cutoff
ECUTRHO = 200  # Ry, GBRV's recommended charge-density cutoff

KPOINT_DENSITY = 4  # Å⁻¹ -> 4x4x3 on the 32-atom cells (Togo: Γ-only in a 4x4x3 = 192-atom cell)
MODEL_TAG = f"{FUNCTIONAL}-{PSEUDOPOTENTIAL_TYPE} {ECUTWFC}-{ECUTRHO}Ry k{KPOINT_DENSITY}"

SCF_UNIT = "pw_scf"
NSCF_UNIT = "pw_nscf"
RELAX_UNIT = "pw_relax"
RELAXATION_SETTINGS = {"forc_conv_thr": 1.9e-3, "nstep": 100}  # 0.05 eV/Å, Togo's convergence target
# Names the relaxation job; the relaxed structure itself is found by content hash.
RELAX_TAG = f"{MODEL_TAG} f{RELAXATION_SETTINGS['forc_conv_thr']}"


## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account

In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load the materials
### 3.1. Load from the uploads folder or the platform, load α-Sn from Standata, and print provenance

In [ ]:
from collections import Counter
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.standata.materials import Materials

materials_by_name = {name: load_material(client, FOLDER, name, ACCOUNT_ID) for name in (PRISTINE_NAME, DEFECTIVE_NAME)}
pristine, pair = materials_by_name[PRISTINE_NAME], materials_by_name[DEFECTIVE_NAME]
sn_reference = Material.create(Materials.get_by_name_first_match(SN_REFERENCE_NAME))
sno2_reference = Material.create(Materials.get_by_name_first_match(SNO2_REFERENCE_NAME))
materials_by_name[SN_REFERENCE_NAME] = sn_reference
materials_by_name[SNO2_REFERENCE_NAME] = sno2_reference

for name, material in materials_by_name.items():
    a, c = material.lattice.a, material.lattice.c
    composition = "".join(f"{e}{n}" for e, n in sorted(Counter(material.basis.elements.values).items()))
    print(f"{name}: {composition}, {material.basis.number_of_atoms} atoms, cell {a:.2f} x {c:.2f} Å")


### 3.2. V_Sn-O_i pair distance

In [ ]:
import math

cartesian_materials = {"pristine": pristine.clone(), "pair": pair.clone()}
sites = {}
for label, material in cartesian_materials.items():
    material.to_cartesian()
    elements = material.basis.elements.values
    coordinates = [tuple(round(value, 4) for value in site) for site in material.basis.coordinates.values]
    for element in ("Sn", "O"):
        sites[(label, element)] = {site for site, species in zip(coordinates, elements) if species == element}

(vacancy_site,) = sites[("pristine", "Sn")] - sites[("pair", "Sn")]
(interstitial_site,) = sites[("pair", "O")] - sites[("pristine", "O")]
v_sn_o_i_distance = math.dist(vacancy_site, interstitial_site)
print(f"V_Sn-O_i distance: {v_sn_o_i_distance:.3f} Å (cf. Fig. 4a)")


### 3.3. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_pristine = get_or_create_material(client, pristine, ACCOUNT_ID)
saved_pair = get_or_create_material(client, pair, ACCOUNT_ID)
saved_sn_reference = get_or_create_material(client, sn_reference, ACCOUNT_ID)
saved_sno2_reference = get_or_create_material(client, sno2_reference, ACCOUNT_ID)


## 4. Configure the shared DFT model and k-grid
### 4.1. DFT model

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.ade.application import Application
from mat3ra.mode import ModelFactory

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

model_config = ModelTreeStandata.get_model_by_parameters(type="dft", subtype="gga", functional=FUNCTIONAL)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = ModelFactory.create(model_config)
print(f"Using application: {app.name}, model: {MODEL_TAG}")


### 4.2. k-grid per material

In [ ]:
from mat3ra.notebooks_utils.workflow import kgrid_from_density

material_objects = {PRISTINE_NAME: pristine, DEFECTIVE_NAME: pair,
                    SN_REFERENCE_NAME: sn_reference, SNO2_REFERENCE_NAME: sno2_reference}

kgrid = {name: kgrid_from_density(material, KPOINT_DENSITY) for name, material in material_objects.items()}
for name, grid in kgrid.items():
    print(f"{name}: k-grid {grid}")


## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}, "
      f"time limit: {TIME_LIMIT}")


## 6. Prerequisite Total Energy jobs

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material

total_energy_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    TOTAL_ENERGY_SEARCH_TERM
)
prerequisite_materials = {PRISTINE_NAME: saved_pristine, SN_REFERENCE_NAME: saved_sn_reference,
                          SNO2_REFERENCE_NAME: saved_sno2_reference}
prerequisite_job_ids = {}
new_job_ids = []
for name, saved_material in prerequisite_materials.items():
    workflow = Workflow.create(total_energy_workflow_config)
    workflow.name = f"Total Energy {name} {MODEL_TAG}"
    workflow.subworkflows[0].model = model
    apply_planewave_cutoffs(workflow, ECUTWFC, ECUTRHO, unit_name=SCF_UNIT)
    apply_scf_kgrid(workflow, kgrid[name], material=material_objects[name])
    job = find_job_for_material(client, saved_material["_id"], workflow.name, ACCOUNT_ID)
    if job is None:
        job = create_job(
            api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
            owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}",
        )
        new_job_ids.append(job["_id"])
        print(f"✅ {name}: created Total Energy job {job['_id']}")
    else:
        print(f"♻️  {name}: reusing existing Total Energy job {job['_id']}")
    prerequisite_job_ids[name] = job["_id"]


In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

if new_job_ids:
    submit_jobs(client.jobs, new_job_ids)
    print(f"✅ Submitted {len(new_job_ids)} prerequisite job(s).")
    await wait_for_jobs_to_finish_async(client.jobs, new_job_ids, poll_interval=POLL_INTERVAL)


## 7. Relax the pair cell (optional)

Runs only if `RELAX`: finds any relaxed version of this structure already on the account,
regardless of who relaxed it or with what -- before spending any compute on a new one.

In [ ]:
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job
from mat3ra.notebooks_utils.core.entity.material.api import find_relaxed_material, get_final_structure_for_job

relaxed_pair = None
if RELAX:
    relax_workflow_name = f"Fixed-cell Relaxation {DEFECTIVE_NAME} {RELAX_TAG}"
    relaxed_pair = find_relaxed_material(client, pair, ACCOUNT_ID)
    if relaxed_pair is not None:
        print(f"♻️  Relaxed pair material: {relaxed_pair.name} ({relaxed_pair.id})")
    else:
        relax_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
            RELAX_WORKFLOW_SEARCH_TERM
        )
        relax_workflow = Workflow.create(relax_workflow_config)
        relax_workflow.name = relax_workflow_name
        relax_workflow.subworkflows[0].model = model
        apply_planewave_cutoffs(relax_workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
        apply_scf_kgrid(relax_workflow, kgrid[DEFECTIVE_NAME], material=pair, unit_name=RELAX_UNIT)
        patch_workflow_qe_input(relax_workflow, {"control": RELAXATION_SETTINGS}, [RELAX_UNIT])

        relax_job = find_job_for_material(
            client, saved_pair["_id"], relax_workflow_name, ACCOUNT_ID,
            statuses=("submitted", "queued", "active", "finished"),
        )
        if relax_job is None:
            relax_job = create_job(
                api_client=client, materials=[saved_pair], workflow=relax_workflow,
                project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
                prefix=f"{relax_workflow_name} {timestamp}",
            )
            submit_jobs(client.jobs, [relax_job["_id"]])
        await wait_for_jobs_to_finish_async(client.jobs, [relax_job["_id"]], poll_interval=POLL_INTERVAL)

        relaxed_material = get_final_structure_for_job(client, relax_job["_id"])
        client.materials.update(relaxed_material.id, {"name": f"{DEFECTIVE_NAME} relaxed"})
        relaxed_pair = Material.create(client.materials.get(relaxed_material.id))
        print(f"✅ Relaxed pair material: {relaxed_pair.name} ({relaxed_pair.id})")

        total_force = get_properties_for_job(client, relax_job["_id"], "total_force")[0]
        print(f"Residual force after relaxation (norm over all atoms): "
              f"{total_force['value']:.4f} {total_force['units']}")


## 8. Density of States job on the pair cell
### 8.1. Configure the workflow

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

dos_material = relaxed_pair.to_dict() if RELAX else saved_pair
dos_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(DOS_SEARCH_TERM)
dos_workflow = Workflow.create(dos_workflow_config)
dos_workflow.name = f"DOS {DEFECTIVE_NAME} {MODEL_TAG}" + (" relaxed" if RELAX else "")
dos_workflow.subworkflows[0].model = model
for unit_name in (SCF_UNIT, NSCF_UNIT):
    apply_planewave_cutoffs(dos_workflow, ECUTWFC, ECUTRHO, unit_name=unit_name)
    apply_scf_kgrid(dos_workflow, kgrid[DEFECTIVE_NAME], material=pair, unit_name=unit_name)

visualize_workflow(dos_workflow)


### 8.2. Create, submit and monitor the job

In [ ]:
dos_job = find_job_for_material(
    client, dos_material["_id"], dos_workflow.name, ACCOUNT_ID, statuses=("submitted", "queued", "active", "finished")
)
if dos_job is None:
    dos_job = create_job(
        api_client=client, materials=[dos_material], workflow=dos_workflow,
        project_id=project_id, owner_id=ACCOUNT_ID, compute=compute.to_dict(),
        prefix=f"{dos_workflow.name} {timestamp}",
    )
    submit_jobs(client.jobs, [dos_job["_id"]])
    print(f"✅ DOS job created and submitted: {dos_job['_id']}")
else:
    print(f"♻️  Reusing existing DOS job: {dos_job['_id']}")
dos_job_id = dos_job["_id"]
await wait_for_jobs_to_finish_async(client.jobs, [dos_job_id], poll_interval=POLL_INTERVAL)


## 9. Retrieve the results
### 9.1. Total energies and formation energy

In [ ]:
e_pristine = get_properties_for_job(client, prerequisite_job_ids[PRISTINE_NAME], property_name="total_energy")[0]["value"]
e_sn = get_properties_for_job(client, prerequisite_job_ids[SN_REFERENCE_NAME], property_name="total_energy")[0]["value"]
e_sno2 = get_properties_for_job(client, prerequisite_job_ids[SNO2_REFERENCE_NAME], property_name="total_energy")[0]["value"]
e_pair = get_properties_for_job(client, dos_job_id, property_name="total_energy")[0]["value"]

formula_units = Counter(pristine.basis.elements.values)["Sn"]
sno2_formula_units = Counter(sno2_reference.basis.elements.values)["Sn"]
mu_sno = e_pristine / formula_units
mu_sno2 = e_sno2 / sno2_formula_units
mu_sn_rich = e_sn / sn_reference.basis.number_of_atoms
mu_o_sn_rich = mu_sno - mu_sn_rich
mu_sn_o_rich = 2 * mu_sno - mu_sno2
mu_o_o_rich = mu_sno2 - mu_sno
e_formation_sn_rich = e_pair - e_pristine + mu_sn_rich - mu_o_sn_rich
e_formation_o_rich = e_pair - e_pristine + mu_sn_o_rich - mu_o_o_rich

print(f"Sn-rich limit: μ_Sn {mu_sn_rich:.4f} eV/atom, μ_O {mu_o_sn_rich:.4f} eV/atom")
print(f"O-rich limit:  μ_Sn {mu_sn_o_rich:.4f} eV/atom, μ_O {mu_o_o_rich:.4f} eV/atom")
print(f"E_f (V_Sn-O_i pair, Sn-rich): {e_formation_sn_rich:.3f} eV")
print(f"E_f (V_Sn-O_i pair, O-rich):  {e_formation_o_rich:.3f} eV")


### 9.2. Density of states and band gap

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

dos_data = get_properties_for_job(client, dos_job_id, property_name="density_of_states")
visualize_properties(dos_data, title=f"DOS {DEFECTIVE_NAME}")

band_gaps_data = get_properties_for_job(client, dos_job_id, property_name="band_gaps")[0]
for gap in band_gaps_data["values"]:
    print(f"{gap['type']} gap: {gap['value']:.3f} {gap['units']}")


## 10. Comparison with Togo et al. (2006)

In [ ]:
TOGO = {"pair_a": 2.3}  # eV, Table II model (a); the paper does not state the μ limit

difference_sn_rich = e_formation_sn_rich - TOGO["pair_a"]
difference_o_rich = e_formation_o_rich - TOGO["pair_a"]
verdict = "yes" if abs(difference_o_rich) <= 0.5 else "no"  # 0.5 eV: cell size, lattice and pseudopotential offsets
config_label = "relaxed defect" if RELAX else "unrelaxed SCF"
print(f"E_f (Togo et al., 2006):      {TOGO['pair_a']:.3f} eV")
print(f"E_f (this notebook, Sn-rich): {e_formation_sn_rich:.3f} eV, difference {difference_sn_rich:+.3f} eV")
print(f"E_f (this notebook, O-rich):  {e_formation_o_rich:.3f} eV, difference {difference_o_rich:+.3f} eV")
print(f"Reproduces Togo et al. (2006): {verdict} ({config_label}, O-rich)")


## References

[1] Togo, A., Oba, F., & Tanaka, I. (2006). First-principles calculations of native defects in tin monoxide. Phys. Rev. B, 74(19), 195128. https://doi.org/10.1103/PhysRevB.74.195128